# Portfolio Risk Engine — le code pas à pas

On étudie un portefeuille de **10 000 €** sur la période fixe du **1er janvier 2019 au 8 octobre 2026 inclus**.
Le taux sans risque est fixé à **2 % par an**. Le notebook reprend les calculs de l'application,
sans son interface Streamlit. Les explications restent courtes pour suivre le raisonnement.

**Pour lancer le notebook :** placer `yahoo_adj_close.csv` dans le même dossier, puis exécuter les cellules dans l'ordre.
Dans le dépôt GitHub, le fichier déjà présent dans `data/` est aussi reconnu.
Les prix téléchargés depuis l'application sont des clôtures ajustées Yahoo Finance.

Si nécessaire, installer les bibliothèques dans une cellule :
`%pip install numpy pandas scipy matplotlib openpyxl`

## 1. Les paramètres

Les poids sont écrits en décimal : 12 % devient `0.12`. Leur somme doit être égale à 1.
Le téléchargement depuis l'application remplit cette cellule avec la dernière analyse validée.
Les scénarios de stress ci-dessous sont les scénarios prédéfinis du projet.

In [ ]:
portfolio_value = 10_000.0
start_date = "2019-01-01"
end_date = "2026-10-08"
risk_free_rate = 0.02
trading_days = 252

tickers = ["AAPL", "NVDA", "MC.PA", "SPY", "TLT", "GLD", "VNQ", "BTC-USD"]
weights = [0.12, 0.08, 0.10, 0.25, 0.15, 0.15, 0.10, 0.05]
currencies = {"AAPL": "USD", "NVDA": "USD", "MC.PA": "EUR", "SPY": "USD",
              "TLT": "USD", "GLD": "USD", "VNQ": "USD", "BTC-USD": "USD"}

confidence_levels = [0.95, 0.99]
horizon = 10
estimation_window = 1250
n_simulations = 100_000
seed = 42
mc_distribution = "normal"  # Choisir "t" pour la loi de Student.
degrees_of_freedom = 5
backtest_window = 500
include_mc_backtest = True


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

weights = pd.Series(weights, index=tickers, dtype=float)
assert weights.ge(0).all() and np.isclose(weights.sum(), 1), "Les poids doivent totaliser 100 %."

holdings = pd.DataFrame({"Poids (%)": weights * 100, "Montant (€)": weights * portfolio_value})
holdings.loc["Total"] = holdings.sum()
display(holdings.round(2))


## 2. Préparer les prix en euros

On conserve les jours où toutes les places boursières du portefeuille ont coté.
Bitcoin et les taux de change sont pris aux mêmes dates ; les week-ends ne deviennent pas des jours boursiers.
Les petits trous sont complétés avec le dernier cours connu, sur trois lignes au maximum.
Pour les actifs en dollars : **prix en EUR = prix en USD / EURUSD**.

In [ ]:
data_file = Path("yahoo_adj_close.csv")
if not data_file.exists():
    data_file = Path("data/yahoo_adj_close.csv")
if not data_file.exists():
    raise FileNotFoundError("Téléchargez le CSV depuis l'application et placez-le à côté du notebook.")

raw = pd.read_csv(data_file, index_col=0, parse_dates=True).sort_index()
raw = raw.loc[start_date:end_date]
fx_columns = sorted({"EUR" + currencies[ticker] + "=X" for ticker in tickers if currencies[ticker] != "EUR"})
raw = raw[tickers + fx_columns]

exchange_tickers = [ticker for ticker in tickers if not ticker.endswith(("-USD", "-EUR"))]
common_dates = raw.index[raw.index.dayofweek < 5]
for ticker in exchange_tickers or tickers:
    common_dates = common_dates.intersection(raw[ticker].dropna().index)

first_common_date = max(raw[ticker].first_valid_index() for ticker in tickers)
common_dates = common_dates[common_dates >= first_common_date]
aligned = raw.ffill(limit=3).loc[common_dates].dropna()

prices = pd.DataFrame(index=aligned.index)
for ticker in tickers:
    currency = currencies[ticker]
    if currency == "EUR":
        prices[ticker] = aligned[ticker]
    else:
        prices[ticker] = aligned[ticker] / aligned["EUR" + currency + "=X"]

print(f"Prix communs : {prices.index.min():%d/%m/%Y} au {prices.index.max():%d/%m/%Y}, {len(prices)} observations.")
display(prices.head().round(2))


## 3. Rendements et fenêtre d'estimation

Le rendement simple sert à calculer le gain ou la perte en euros. Le rendement du portefeuille
est la somme des rendements pondérés : on suppose donc un rééquilibrage quotidien.
Comme dans l'application, les rendements logarithmiques servent aux statistiques annualisées.
La VaR utilise seulement les derniers jours de la fenêtre d'estimation choisie.

In [ ]:
returns = prices.pct_change(fill_method=None).dropna()
log_returns = np.log(prices).diff().dropna()
portfolio_returns = returns @ weights

assert len(returns) >= estimation_window, "La fenêtre d'estimation dépasse l'historique disponible."
estimation = returns.tail(estimation_window)
portfolio_estimation = estimation @ weights
mean_returns = estimation.mean().to_numpy()
covariance = estimation.cov().to_numpy()
weight_array = weights.to_numpy()

print(f"Rendements : {returns.index.min():%d/%m/%Y} au {returns.index.max():%d/%m/%Y}.")
print(f"Estimation du risque : {estimation.index.min():%d/%m/%Y} au {estimation.index.max():%d/%m/%Y}.")


## 4. Performance et ratio de Sharpe

On retient 252 jours de bourse par an. Le Sharpe est calculé avec le **taux sans risque constant de 2 %** :
`(rendement annuel − 0.02) / volatilité annuelle`.
Le drawdown mesure la baisse depuis le plus haut déjà atteint.

In [ ]:
all_returns = returns.assign(Portfolio=portfolio_returns)
all_logs = log_returns.assign(Portfolio=np.log1p(portfolio_returns))

annual_return = all_logs.mean() * trading_days
annual_volatility = all_logs.std(ddof=1) * np.sqrt(trading_days)
wealth = (1 + all_returns).cumprod()
drawdown = wealth / wealth.cummax() - 1

performance = pd.DataFrame({
    "Ann. return": annual_return,
    "Ann. volatility": annual_volatility,
    "Sharpe": (annual_return - risk_free_rate) / annual_volatility,
    "Skewness": all_returns.skew(),
    "Excess kurtosis": all_returns.kurt(),
    "Worst day": all_returns.min(),
    "Best day": all_returns.max(),
    "Max drawdown": drawdown.min(),
})
display(performance.round(4))

drawdown["Portfolio"].plot(figsize=(10, 3), title="Drawdown du portefeuille", ylabel="Baisse depuis le plus haut")
plt.show()


## 5. Corrélation et diversification

Des actifs moins corrélés permettent de réduire le risque global.
Le ratio de diversification compare la moyenne pondérée des volatilités individuelles à celle du portefeuille.
On regarde aussi si les corrélations changent entre les périodes calmes et les crises.

In [ ]:
asset_volatilities = np.sqrt(np.diag(covariance))
portfolio_mean = float(weight_array @ mean_returns)
portfolio_volatility = float(np.sqrt(weight_array @ covariance @ weight_array))
diversification_ratio = float(weight_array @ asset_volatilities / portfolio_volatility)
display(estimation.corr().round(2))
print(f"Ratio de diversification : {diversification_ratio:.2f}")

regimes = {
    "Calme 2019": ("2019-01-01", "2019-12-31"),
    "Covid": ("2020-02-19", "2020-04-30"),
    "Choc de taux 2022": ("2022-01-03", "2022-10-12"),
    "Tarifs 2025": ("2025-02-19", "2025-04-30"),
}
regime_rows = []
for name, (begin, end) in regimes.items():
    period_returns = returns.loc[begin:end]
    if len(period_returns) < 2:
        continue
    correlation = period_returns.corr().to_numpy()
    n_assets = len(tickers)
    average_correlation = (correlation.sum() - n_assets) / (n_assets * (n_assets - 1))
    regime_rows.append({"Période": name, "Corrélation moyenne": average_correlation})
display(pd.DataFrame(regime_rows).set_index("Période").round(3))


## 6. VaR et Expected Shortfall historiques

À 99 %, la VaR marque le début du 1 % des pertes les plus fortes. L'ES est la perte moyenne au-delà de ce seuil.
Les deux sont exprimées en euros ; une valeur positive correspond à une perte.
Sur plusieurs jours, on compose les rendements quotidiens sur des fenêtres qui se chevauchent.

In [ ]:
def var_es(pnl, confidence):
    pnl = np.asarray(pnl)
    threshold = np.quantile(pnl, 1 - confidence)
    value_at_risk = -threshold
    expected_shortfall = -pnl[pnl <= threshold].mean()
    return float(value_at_risk), float(expected_shortfall)


def historical_risk(asset_returns, portfolio_weights, confidence, days=1):
    daily_returns = asset_returns @ portfolio_weights
    if days > 1:
        daily_returns = (1 + daily_returns).rolling(days).apply(np.prod, raw=True).dropna() - 1
    return var_es(portfolio_value * daily_returns, confidence)


historical_var, historical_es = historical_risk(estimation, weights, max(confidence_levels))
print(f"VaR historique : {historical_var:.2f} € ; ES historique : {historical_es:.2f} €.")


## 7. VaR et ES paramétriques

Ici, on suppose des rendements normalement distribués. Il suffit de la moyenne et de la covariance estimées.
Pour plusieurs jours, la moyenne est multipliée par l'horizon et la volatilité par sa racine carrée.

In [ ]:
def parametric_risk(mean, cov, portfolio_weights, confidence, days=1):
    average = float(portfolio_weights @ mean) * days
    volatility = float(np.sqrt(portfolio_weights @ cov @ portfolio_weights)) * np.sqrt(days)
    z = stats.norm.ppf(confidence)
    value_at_risk = portfolio_value * (z * volatility - average)
    expected_shortfall = portfolio_value * (volatility * stats.norm.pdf(z) / (1 - confidence) - average)
    return value_at_risk, expected_shortfall


parametric_var, parametric_es = parametric_risk(mean_returns, covariance, weight_array, max(confidence_levels))
print(f"VaR normale : {parametric_var:.2f} € ; ES normale : {parametric_es:.2f} €.")


## 8. Monte Carlo : normale ou Student-t

On simule des rendements en conservant les corrélations entre actifs.
La loi normale donne la courbe en cloche habituelle. Student-t génère plus souvent des valeurs extrêmes.
Ses **degrés de liberté ν sont fixés à 5** : plus ν est petit, plus les queues sont épaisses.
Le facteur de correction dans le code garde la même covariance pour comparer les deux modèles.

In [ ]:
def simulate_returns(mean, cov, n_draws, days=1, distribution="normal", random_seed=42):
    rng = np.random.default_rng(random_seed)
    try:
        cholesky = np.linalg.cholesky(cov)
    except np.linalg.LinAlgError:
        # Une très petite correction suffit si la matrice est presque singulière.
        cholesky = np.linalg.cholesky(cov + np.eye(len(mean)) * 1e-12)
    gross_returns = np.ones((n_draws, len(mean)))
    for day in range(days):
        shocks = rng.standard_normal((n_draws, len(mean)))
        if distribution == "t":
            chi_square = rng.chisquare(degrees_of_freedom, size=(n_draws, 1))
            shocks = shocks / np.sqrt(chi_square / degrees_of_freedom)
            shocks *= np.sqrt((degrees_of_freedom - 2) / degrees_of_freedom)
        gross_returns *= 1 + mean + shocks @ cholesky.T
    return gross_returns - 1


risk_rows = []
for days in [1, horizon]:
    simulations = simulate_returns(mean_returns, covariance, n_simulations, days, mc_distribution, seed)
    simulated_pnl = portfolio_value * (simulations @ weight_array)
    for confidence in confidence_levels:
        estimates = {
            "Historical": historical_risk(estimation, weights, confidence, days),
            "Parametric": parametric_risk(mean_returns, covariance, weight_array, confidence, days),
            "Monte Carlo": var_es(simulated_pnl, confidence),
        }
        if days > 1:
            one_day = historical_risk(estimation, weights, confidence)
            estimates["Historical (√h rule)"] = np.array(one_day) * np.sqrt(days)
        for method, (var, es) in estimates.items():
            risk_rows.append({"Method": method, "Horizon": days, "Confidence": confidence, "VaR": var, "ES": es})
risk_results = pd.DataFrame(risk_rows)
display(risk_results.round(2))


## 9. Quelques variantes et la stabilité de Monte Carlo

La Student-t paramétrique estime ν sur les rendements du portefeuille : ce ν peut donc être différent de 5.
Cornish-Fisher tient compte de l'asymétrie et de la kurtosis. EWMA donne plus de poids aux observations récentes.
Enfin, répéter Monte Carlo montre que le résultat dépend aussi du nombre de tirages.

In [ ]:
fitted_nu = stats.t.fit(portfolio_estimation)[0]
skewness = stats.skew(portfolio_estimation)
excess_kurtosis = stats.kurtosis(portfolio_estimation)
ewma_variance = np.var(portfolio_estimation.iloc[:30])
for daily_return in portfolio_estimation:
    ewma_variance = 0.94 * ewma_variance + 0.06 * daily_return ** 2

student_draws = simulate_returns(mean_returns, covariance, n_simulations, 1, "t", seed)
variant_rows = []
for confidence in confidence_levels:
    z = stats.norm.ppf(confidence)
    student_z = stats.t.ppf(confidence, fitted_nu) * np.sqrt((fitted_nu - 2) / fitted_nu) if fitted_nu > 2 else np.nan
    cf_z = z + (z ** 2 - 1) * skewness / 6 + (z ** 3 - 3 * z) * excess_kurtosis / 24
    cf_z -= (2 * z ** 3 - 5 * z) * skewness ** 2 / 36
    variant_rows.append({
        "Confidence": confidence,
        "Normal": (z * portfolio_volatility - portfolio_mean) * portfolio_value,
        "Student-t ajustée": (student_z * portfolio_volatility - portfolio_mean) * portfolio_value,
        "Cornish-Fisher": (cf_z * portfolio_volatility - portfolio_mean) * portfolio_value,
        "EWMA": (z * np.sqrt(ewma_variance) - portfolio_mean) * portfolio_value,
        "MC Student-t (ν = 5)": var_es(portfolio_value * (student_draws @ weight_array), confidence)[0],
    })
variants = pd.DataFrame(variant_rows).set_index("Confidence")
print(f"Degrés de liberté estimés : {fitted_nu:.2f}")
display(variants.round(2))


In [ ]:
convergence_rows = []
for n_draws in [1_000, 10_000, 100_000]:
    estimates = []
    for run in range(25):
        draws = simulate_returns(mean_returns, covariance, n_draws, random_seed=seed + run)
        estimates.append(var_es(portfolio_value * (draws @ weight_array), 0.99)[0])
    convergence_rows.append({"Tirages": n_draws, "VaR moyenne": np.mean(estimates),
                             "Écart-type entre essais": np.std(estimates, ddof=1)})
display(pd.DataFrame(convergence_rows).set_index("Tirages").round(2))


## 10. Contribution de chaque actif au risque

La contribution à la VaR normale dépend de la covariance avec le portefeuille.
Pour l'ES historique, on regarde les pertes de chaque position pendant les journées qui dépassent la VaR.
Les contributions s'additionnent pour retrouver le risque total.

In [ ]:
confidence = max(confidence_levels)
component_var = portfolio_value * weight_array * (
    stats.norm.ppf(confidence) * (covariance @ weight_array) / portfolio_volatility - mean_returns
)
historical_pnl = portfolio_value * portfolio_estimation
threshold = np.quantile(historical_pnl, 1 - confidence)
tail_returns = estimation.loc[historical_pnl <= threshold]
component_es = -portfolio_value * tail_returns.mul(weights).mean()

contributions = pd.DataFrame({"Poids": weights, "VaR normale (€)": component_var, "ES historique (€)": component_es})
contributions.loc["Total"] = contributions.sum()
display(contributions.round(2))


## 11. Backtest : comparer la VaR aux pertes suivantes

Pour chaque jour, on utilise uniquement les jours précédents pour estimer la VaR.
Une exception arrive quand la perte réelle dépasse cette prévision.
Le Monte Carlo de ce backtest reste **normal**, comme dans l'application ; il utilise 5 000 tirages par jour.
Les tests statistiques suivants vérifient le nombre d'exceptions et leur indépendance.

In [ ]:
def run_backtest(confidence):
    z = stats.norm.ppf(confidence)
    ewma_variance = portfolio_returns.iloc[:backtest_window].var(ddof=0)
    rows = []
    for day in range(backtest_window, len(returns)):
        sample = returns.iloc[day - backtest_window:day]
        daily_sample = portfolio_returns.iloc[day - backtest_window:day]
        mean = sample.mean().to_numpy()
        cov = sample.cov().to_numpy()
        volatility = np.sqrt(weight_array @ cov @ weight_array)
        ewma_variance = 0.94 * ewma_variance + 0.06 * portfolio_returns.iloc[day - 1] ** 2
        row = {"Date": returns.index[day], "P&L": portfolio_value * portfolio_returns.iloc[day],
               "Historical": -daily_sample.quantile(1 - confidence) * portfolio_value,
               "Parametric": (z * volatility - weight_array @ mean) * portfolio_value,
               "EWMA parametric": (z * np.sqrt(ewma_variance) - weight_array @ mean) * portfolio_value}
        if include_mc_backtest:
            draws = simulate_returns(mean, cov, 5_000, random_seed=seed + day)
            row["Monte Carlo"] = var_es(portfolio_value * (draws @ weight_array), confidence)[0]
        rows.append(row)
    return pd.DataFrame(rows).set_index("Date")


backtests = {confidence: run_backtest(confidence) for confidence in confidence_levels}


In [ ]:
def log_likelihood(probability, zeros, ones):
    # xlogy gère aussi les cas sans exception, sans faire log(0).
    from scipy.special import xlogy
    return xlogy(zeros, 1 - probability) + xlogy(ones, probability)


backtest_rows = []
for confidence, backtest in backtests.items():
    for method in backtest.columns.drop("P&L"):
        exceptions = (-backtest["P&L"] > backtest[method]).astype(int)
        n_days = len(exceptions)
        n_exceptions = int(exceptions.sum())
        expected_rate = 1 - confidence
        actual_rate = n_exceptions / n_days

        kupiec_lr = -2 * (log_likelihood(expected_rate, n_days - n_exceptions, n_exceptions)
                          - log_likelihood(actual_rate, n_days - n_exceptions, n_exceptions))

        previous = exceptions.to_numpy()[:-1]
        following = exceptions.to_numpy()[1:]
        n00 = int(((previous == 0) & (following == 0)).sum())
        n01 = int(((previous == 0) & (following == 1)).sum())
        n10 = int(((previous == 1) & (following == 0)).sum())
        n11 = int(((previous == 1) & (following == 1)).sum())
        p01 = n01 / max(n00 + n01, 1)
        p11 = n11 / max(n10 + n11, 1)
        probability = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)
        independence_lr = -2 * (log_likelihood(probability, n00 + n10, n01 + n11)
                                - log_likelihood(p01, n00, n01) - log_likelihood(p11, n10, n11))
        backtest_rows.append({"Confidence": confidence, "Model": method, "Days": n_days,
                              "Expected": n_days * expected_rate, "Exceptions": n_exceptions, "Rate": actual_rate,
                              "Kupiec p": stats.chi2.sf(kupiec_lr, 1),
                              "Christoffersen p": stats.chi2.sf(independence_lr, 1)})
backtest_summary = pd.DataFrame(backtest_rows)
display(backtest_summary.round(4))


Une p-value inférieure à 5 % conduit à rejeter l'hypothèse testée au seuil de 5 %.
Ne pas la rejeter ne prouve pas que le modèle est parfait.
Le graphique affiche les pertes réelles et la limite de VaR historique, avec la même convention de signe.

In [ ]:
backtest = backtests[max(confidence_levels)]
ax = backtest["P&L"].plot(figsize=(11, 3), alpha=0.6, label="P&L réel")
(-backtest["Historical"]).plot(ax=ax, label="Limite de VaR historique", color="red")
ax.set_ylabel("Euros")
ax.legend()
plt.show()


## 12. Stress tests historiques et hypothétiques

On rejoue les crises couvertes par nos dates avec les poids actuels, en buy-and-hold pendant chaque épisode.
Ces scénarios n'ont pas de probabilité associée. La crise de 2008 est hors de l'intervalle de l'étude.
On ajoute un choc choisi par actif, puis un recul du marché transmis par les bêtas.

In [ ]:
episodes = {
    "Covid": ("2020-02-19", "2020-03-23"),
    "Choc de taux 2022": ("2022-01-03", "2022-10-12"),
    "Tarifs 2025": ("2025-02-19", "2025-04-08"),
}
stress_rows = []
for name, (begin, end) in episodes.items():
    episode_prices = prices.loc[begin:end]
    if pd.Timestamp(begin) < prices.index.min() or len(episode_prices) < 2:
        continue
    asset_shocks = episode_prices.iloc[-1] / episode_prices.iloc[0] - 1
    loss = -portfolio_value * float(weights @ asset_shocks)
    stress_rows.append({"Scenario": name, "Loss": loss, "Loss %": loss / portfolio_value})

default_shocks = {"AAPL": -0.25, "NVDA": -0.35, "MC.PA": -0.15, "SPY": -0.18,
                  "TLT": 0.04, "GLD": 0.05, "VNQ": -0.15, "BTC-USD": -0.45}
shocks = pd.Series({ticker: default_shocks.get(ticker, -0.20) for ticker in tickers})
market_ticker = "SPY" if "SPY" in tickers else tickers[0]
betas = estimation.cov()[market_ticker] / estimation[market_ticker].var()

for name, asset_shocks in {"Chocs par actif": shocks, "Marché -20 % via les bêtas": betas * -0.20}.items():
    loss = -portfolio_value * float(weights @ asset_shocks)
    stress_rows.append({"Scenario": name, "Loss": loss, "Loss %": loss / portfolio_value})
stress_results = pd.DataFrame(stress_rows).set_index("Scenario")
display(stress_results.round(3))


## 13. Stress de corrélation et de volatilité

On fixe les corrélations entre actifs à 0,90 et/ou on double leurs volatilités.
On recalcule ensuite la VaR et l'ES normales à un jour avec ces nouvelles covariances.

In [ ]:
stressed_correlation = np.full((len(tickers), len(tickers)), 0.90)
np.fill_diagonal(stressed_correlation, 1)
stressed_covariance = np.outer(asset_volatilities, asset_volatilities) * stressed_correlation
covariance_scenarios = {"Base": covariance, "Corrélations à 0,90": stressed_covariance,
                        "Volatilités x2": covariance * 4, "Corrélations et volatilités": stressed_covariance * 4}
covariance_rows = []
for name, cov in covariance_scenarios.items():
    var, es = parametric_risk(mean_returns, cov, weight_array, max(confidence_levels))
    covariance_rows.append({"Scenario": name, "Annual volatility": np.sqrt(weight_array @ cov @ weight_array * trading_days),
                            "VaR": var, "ES": es})
covariance_results = pd.DataFrame(covariance_rows).set_index("Scenario")
display(covariance_results.round(3))


## 14. Reverse stress test et concentration

Le reverse stress cherche un choc qui provoque une perte de 20 %.
Le vecteur `joint_shock` donne le choc le moins éloigné selon notre modèle de covariance ; cela reste une hypothèse de modèle.
Ensuite, on transfère progressivement les poids vers l'actif le plus volatil pour observer la concentration.

In [ ]:
loss_limit = 0.20
joint_shock = -loss_limit * (covariance @ weight_array) / (weight_array @ covariance @ weight_array)
portfolio_beta = float(weights @ betas)
market_shock = -loss_limit / portfolio_beta if portfolio_beta != 0 else np.nan
print(f"Perte cible : {loss_limit * portfolio_value:.0f} € ; choc de marché via les bêtas : {market_shock:.1%}.")
display(pd.Series(joint_shock, index=tickers, name="Choc conjoint").round(4))

concentrated_asset = performance.drop("Portfolio")["Ann. volatility"].idxmax()
target_weights = pd.Series(0.0, index=tickers)
target_weights[concentrated_asset] = 1
concentration_rows = []
for fraction in np.linspace(0, 1, 11):
    new_weights = (1 - fraction) * weights + fraction * target_weights
    var, es = historical_risk(estimation, new_weights, 0.99)
    concentration_rows.append({"Poids dans " + concentrated_asset: new_weights[concentrated_asset],
                               "Nombre effectif de positions": 1 / (new_weights ** 2).sum(), "VaR 99%": var, "ES 99%": es})
concentration_results = pd.DataFrame(concentration_rows)
display(concentration_results.round(3))

equal_weights = pd.Series(1 / len(tickers), index=tickers)
equal_var, equal_es = historical_risk(estimation, equal_weights, 0.99)
print(f"Portefeuille équipondéré : VaR 99 % = {equal_var:.2f} € ; ES 99 % = {equal_es:.2f} €.")


## 15. Garder les résultats

On rassemble les principaux tableaux dans un fichier Excel pour les relire ou les utiliser dans le rapport.
Les résultats en euros reposent sur le capital fixe de 10 000 € ; les pourcentages ne dépendent pas de ce montant.

In [ ]:
with pd.ExcelWriter("notebook_results.xlsx", engine="openpyxl") as writer:
    holdings.to_excel(writer, sheet_name="Portfolio")
    performance.to_excel(writer, sheet_name="Performance")
    risk_results.to_excel(writer, sheet_name="VaR ES", index=False)
    variants.to_excel(writer, sheet_name="Variants")
    contributions.to_excel(writer, sheet_name="Contributions")
    backtest_summary.to_excel(writer, sheet_name="Backtesting", index=False)
    stress_results.to_excel(writer, sheet_name="Stress tests")
    covariance_results.to_excel(writer, sheet_name="Covariance stress")
    concentration_results.to_excel(writer, sheet_name="Concentration", index=False)
print("Résultats enregistrés dans notebook_results.xlsx.")
